<a href="https://colab.research.google.com/github/nrapendra-rao/Nrapendra-Rao_25SCS1003001338_IILM-GN/blob/main/AI_Text_Summarizer_code_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
\


# -------------------- INSTALL DEPENDENCIES --------------------
# (For Colab only — remove if running locally)
# !pip install pdfplumber gradio transformers sentencepiece reportlab rouge-score --quiet

# -------------------- IMPORT LIBRARIES --------------------
import gradio as gr
import pdfplumber
from transformers import pipeline
from reportlab.pdfgen import canvas
from rouge_score import rouge_scorer

# -------------------- LOAD SUMMARIZATION MODEL --------------------
abstractive_model = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",
    tokenizer="sshleifer/distilbart-cnn-12-6"
)

# -------------------- PDF TEXT EXTRACTION --------------------
def extract_text(file_obj):
    if file_obj is None:
        return ""
    text = ""
    try:
        with pdfplumber.open(file_obj.name) as pdf:
            for page in pdf.pages:
                pg = page.extract_text()
                if pg:
                    text += pg + "\n"
    except:
        return ""
    return text.strip()

# -------------------- EXTRACTIVE SUMMARY --------------------
def extractive_summary(text):
    sentences = text.split(".")
    top = sentences[:5]
    return ". ".join(top).strip() + "."

# -------------------- ABSTRACTIVE SUMMARY --------------------
def abstractive_summary(text):
    text = text[:900]
    result = abstractive_model(
        text,
        max_length=220,
        min_length=80,
        do_sample=False
    )
    return result[0]["summary_text"]

# -------------------- ROUGE SCORE CALCULATOR --------------------
def get_rouge_scores(reference, summary):
    scorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
    scores = scorer.score(reference, summary)

    result = {
        "ROUGE-1 Precision": round(scores['rouge1'].precision, 4),
        "ROUGE-1 Recall": round(scores['rouge1'].recall, 4),
        "ROUGE-1 F1 Score": round(scores['rouge1'].fmeasure, 4),
        "ROUGE-2 Precision": round(scores['rouge2'].precision, 4),
        "ROUGE-2 Recall": round(scores['rouge2'].recall, 4),
        "ROUGE-2 F1 Score": round(scores['rouge2'].fmeasure, 4),
        "ROUGE-L Precision": round(scores['rougeL'].precision, 4),
        "ROUGE-L Recall": round(scores['rougeL'].recall, 4),
        "ROUGE-L F1 Score": round(scores['rougeL'].fmeasure, 4)
    }
    return "\n".join([f"{k}: {v}" for k,v in result.items()])

# -------------------- PDF DOWNLOAD GENERATOR --------------------
def create_summary_pdf(extractive, abstractive):
    filename = "final_summary.pdf"
    c = canvas.Canvas(filename)
    c.setFont("Helvetica-Bold", 16)
    c.drawString(30, 800, "Summary Report")

    c.setFont("Helvetica-Bold", 12)
    c.drawString(30, 770, "Extractive Summary:")
    c.setFont("Helvetica", 10)

    y = 750
    for line in extractive.split(". "):
        c.drawString(30, y, line.strip())
        y -= 15

    c.setFont("Helvetica-Bold", 12)
    c.drawString(30, y - 20, "Abstractive Summary:")
    c.setFont("Helvetica", 10)
    y -= 40

    for line in abstractive.split(". "):
        c.drawString(30, y, line.strip())
        y -= 15

    c.save()
    return filename

# -------------------- MAIN SUMMARIZER FUNCTION --------------------
def process_pdf(file_obj):
    if file_obj is None:
        return "⚠ No PDF uploaded.", "", None, None, None, ""

    text = extract_text(file_obj)
    if not text.strip():
        return "⚠ Could not read text from PDF.", "", None, None, None, ""

    ext = extractive_summary(text)
    abs = abstractive_summary(text)

    with open("extractive.txt", "w") as f:
        f.write(ext)

    with open("abstractive.txt", "w") as f:
        f.write(abs)

    pdf_file = create_summary_pdf(ext, abs)
    rouge_scores = get_rouge_scores(text, abs)

    return ext, abs, "extractive.txt", "abstractive.txt", pdf_file, rouge_scores

# -------------------- GRADIO WEB APP --------------------
with gr.Blocks(title="PDF Summarizer Pro") as app:

    gr.Markdown("# 📄 **Advanced PDF Summarizer Pro**")

    pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
    generate_btn = gr.Button("✨ Generate Summaries", variant="primary")

    extractive_box = gr.Textbox(label="🟦 Extractive Summary", lines=8)
    abstractive_box = gr.Textbox(label="🟧 Abstractive Summary", lines=8)

    download_extractive = gr.File(label="⬇ Download Extractive Summary")
    download_abstractive = gr.File(label="⬇ Download Abstractive Summary")
    download_pdf = gr.File(label="⬇ Download Full Summary PDF")

    rouge_output = gr.Textbox(label="📊 ROUGE Scores", lines=12)

    generate_btn.click(
        process_pdf,
        inputs=pdf_input,
        outputs=[
            extractive_box,
            abstractive_box,
            download_extractive,
            download_abstractive,
            download_pdf,
            rouge_output
        ]
    )

app.launch()
